This is for cut the training videos to small clips for traning.

In [ ]:
import os
import subprocess
import json

def get_video_fps(input_path):
    """
    获取视频的帧率
    """
    command = [
        "ffprobe",
        "-v", "quiet",
        "-print_format", "json",
        "-show_streams",
        "-select_streams", "v:0",
        input_path
    ]
    
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        data = json.loads(result.stdout)
        
        # 获取帧率
        fps_str = data['streams'][0]['r_frame_rate']
        # 处理分数形式的帧率 (如 "30000/1001")
        if '/' in fps_str:
            num, den = map(int, fps_str.split('/'))
            fps = num / den
        else:
            fps = float(fps_str)
        
        return fps
    except Exception as e:
        print(f"获取帧率失败: {e}")
        return None

def split_video_by_frames(input_path, output_dir, frames_per_segment=240):
    """
    按帧数分割视频，确保每个片段包含指定数量的帧
    
    参数:
    - input_path: 原始视频路径
    - output_dir: 输出分段视频保存的目录
    - frames_per_segment: 每个视频片段的帧数
    """
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    
    # 获取原始视频的帧率
    original_fps = get_video_fps(input_path)
    if original_fps is None:
        print("无法获取视频帧率，使用默认值30fps")
        original_fps = 29.97
    
    print(f"原始视频帧率: {original_fps:.2f}fps")
    
    # 计算每个片段的时长（秒）
    segment_duration = frames_per_segment / original_fps
    print(f"每个片段时长: {segment_duration:.3f}秒")
    print(f"每个片段帧数: {frames_per_segment}帧")
    print(f"转换为25fps后每个片段帧数: {frames_per_segment * 25 / original_fps:.1f}帧")
    
    output_pattern = os.path.join(output_dir, f"{base_name}_%03d.mp4")
    
    command = [
        "ffmpeg",
        "-i", input_path,
        "-c:v", "libx264",  # 重新编码视频
        "-c:a", "aac",      # 重新编码音频
        "-f", "segment",
        "-segment_time", str(segment_duration),
        "-force_key_frames", f"expr:gte(t,n_forced*{segment_duration})",  # 强制在指定时间点插入关键帧
        "-reset_timestamps", "1",
        "-avoid_negative_ts", "make_zero",
        "-y",  # 覆盖输出文件
        output_pattern
    ]
    
    try:
        subprocess.run(command, check=True)
        print("分割完成！输出目录:", output_dir)
        
        # 验证输出文件
        output_files = [f for f in os.listdir(output_dir) if f.endswith('.mp4')]
        print(f"生成了 {len(output_files)} 个视频片段")
        
    except subprocess.CalledProcessError as e:
        print("分割出错:", e)

# 示例用法
if __name__ == "__main__":
    input_video_path = "/Users/yuqiliang/Documents/UCLPhD/pilot_study/vid/CamdenTown1_EQR_720p_190502.mov"
    output_folder = "/Users/yuqiliang/Documents/UCLPhD/pilot_study/output_split"
    
    # 按240帧分割视频
    print("=== 开始分割视频 ===")
    split_video_by_frames(input_video_path, output_folder, frames_per_segment=240)
    
    print("\n=== 分割完成 ===")
    print(f"分割后的视频保存在: {output_folder}")